# Inference for numerical data

## North Carolina births

In 2004, the state of North Carolina released a large data set containing information on births recorded in this state. This data set is useful to researchers studying the relation between habits and practices of expectant mothers and the birth of their children. We will work with a random sample of observations from this data set.

## Exploratory analysis

Load the `nc` data set into our notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.stats as stats
import io
import requests
from plotnine import *

df_url = 'https://raw.githubusercontent.com/akmand/datasets/master/openintro/nc.csv'
url_content = requests.get(df_url, verify=False).content
nc = pd.read_csv(io.StringIO(url_content.decode('utf-8')))
nc = nc.dropna().reset_index(drop = True)
nc.head()

We have observations on 13 different variables, some categorical and some numerical. The meaning of each variable is as follows.

| variable         | description |
| ---------------- | ------------|
| `fage`           | father's age in years. |
| `mage`           | mother's age in years. |
| `mature`         | maturity status of mother. |
| `weeks`          | length of pregnancy in weeks. |
| `premie`         | whether the birth was classified as premature (premie) or full-term. |
| `visits`         | number of hospital visits during pregnancy. |
| `marital`        | whether mother is `married` or `not married` at birth. |
| `gained`         | weight gained by mother during pregnancy in pounds. |
| `weight`         | weight of the baby at birth in pounds. |
| `lowbirthweight` | whether baby was classified as low birthweight (`low`) or not (`not low`). |
| `gender`         | gender of the baby, `female` or `male`. |
| `habit`          | status of the mother as a `nonsmoker` or a `smoker`. |
| `whitemom`       | whether mom is `white` or `not white`. |

### Exercise 1
- What are the cases (or observations) in this data set?
- How many cases are there in our sample?

```
Hint: Python has ways of telling you the shape of a dataframe. Refer to
Lab 1 if you're stuck.
```

In [ ]:
# Determine the number cases in the dataset.



As a first step in the analysis, we should consider summaries of the data. This can be done using the `.describe()` function.

In [ ]:
nc.describe()

Consider the possible relationship between a mother's smoking habit and the weight of her baby. Plotting the data is a useful first step because it helps us quickly visualize trends, identify strong associations, and develop research questions.

### Exercise 2

- Make a side-by-side boxplot of <code>habit</code> and <code>weight</code>.
- What does the plot highlight about the relationship between these two variables?

```
Hint: Refer to Lab 1 for a reminder of how to make boxplot in Python.
```

In [ ]:
# Make a side-by-side boxplot of `habit` and `weight`.



The box plots show how the medians of the two distributions compare, but we can also compare the means of the distributions using `mean()`. First, we'll create a dataframe consisting of just the smokers.

In [ ]:
smokers = nc[nc['habit'] == 'smoker']

smokers.head()

Then, we can use `.mean()` to find the mean baby weight from this data.

In [ ]:
mean_weight_smokers = smokers['weight'].mean()

print(mean_weight_smokers)

### Exercise 3

- Create a dataframe consisting of just the nonsmokers and call it `nonsmokers`. Then, find the mean of the `weight` variable from this dataset.

- Compare the mean baby weight from the `smokers` dataset to the `nonsmokers` dataset.

In [ ]:
nonsmokers = nc[nc['habit'] == ???] # Fill in the code!

nonsmokers.head()

In [ ]:
mean_weight_nonsmokers = ??? # Fill in the code!

print(mean_weight_nonsmokers)

There is an observed difference, but is this difference statistically significant? In order to answer this question we will conduct a hypothesis test.

## Preparing for inference

### Exercise 4

Check if the conditions necessary for inference are satisfied.

```
Hint: You will need to obtain sample sizes to check the conditions.
```

In [ ]:
# Run this code to make histgrams of `weight` by `habit`.

(
    ggplot(nc) +
    aes(x = 'weight') +
    geom_histogram() +
    facet_wrap('habit')
)

In [ ]:
# Run this code to find the number of smokers and save it as `n_smokers`.

n_smokers = len(smokers['weight'])

print(n_smokers)

In [ ]:
# Fill in the code to find the number of nonsmokers and save it as `n_nonsmokers`.

n_nonsmokers = ???

print(n_nonsmokers)

### Exercise 5

Write the hypotheses for testing if the average weights of babies born to smoking and non-smoking mothers are different.

- $H_0$:
- $H_A$:

### Exercise 6

The code in Exercise 8 of the next section runs a T-test for a difference of means. Would it be reasonable, in this case, to use a Z-test instead? Why or why not?

```
Hint: Consider the degrees of freedom for the corresponding t-distribution.
```

## Hypothesis testing

We will now conduct hypothesis tests to investigate if the average weights of babies born to smoking and non-smoking mothers are different.

### Exercise 7

- Fill in the code below to onstruct a 95% confidence interval for the difference between the weights of babies born to smoking and non-smoking mothers.

- From this confidence interval, draw an appropriate conclusion regarding the hypotheses.

In [ ]:
# Find a 95% confidence interval for the difference of mean baby weights
# between smokers and nonsmokers.

confidence_level = ??? # Fill in the confidence level (as a decimal).

sd_weight_smokers = ??? # Compute the standard deviation of the baby weights of the smokers.
sd_weight_nonsmokers = ??? # Compute the standard deviation of the baby weights of the nonsmokers.

degrees_of_freedom = min(n_smokers - 1, n_nonsmokers - 1)
t_star = stats.t.ppf(1 - (1 - confidence_level) / 2, degrees_of_freedom)

mean_diff = ??? # Find the difference of the mean baby weights between the two groups.

SE = np.sqrt(sd_weight_smokers**2/n_smokers + sd_weight_nonsmokers**2/n_nonsmokers)

lower_bound = mean_diff - t_star * SE
upper_bound = mean_diff + t_star * SE

print(f'{confidence_level*100}% Confidence Interval: ({lower_bound}, {upper_bound})')

### Exercise 8

Run the code below to perform a T-test on the difference of mean baby weights between smokers and nonsmokers. Interpret the output of the hypothesis test code in context.

In [ ]:
# Run this code to perform a T-test for difference of mean baby weights
# between smokers and nonsmokers.

alpha = 0.05

degrees_of_freedom = min(n_smokers - 1, n_nonsmokers - 1)

t_star = stats.t.ppf(1 - alpha / 2, degrees_of_freedom)

null_diff = 0
mean_diff = mean_weight_smokers - mean_weight_nonsmokers

SE = np.sqrt(sd_weight_smokers**2/n_smokers + sd_weight_nonsmokers**2/n_nonsmokers)

T_score = (mean_diff - null_diff) / SE

p_value = 2 * (1 - stats.t.cdf(abs(T_score), degrees_of_freedom))

print(f'T-score = {T_score}')
print(f'p-value = {p_value}')

if p_value < alpha:
    print("Reject null hypothesis")
else:
    print("Fail to reject null hypothesis")

### Exercise 9

Conduct a hypothesis test to evaluate whether the average length of a pregnancy of younger mothers is different than the average length of pregnancy of mature mothers. Use a significance level of $\alpha = 0.05$.

Your test should include each of the following steps.

- Identify the null and alternative hypotheses.

- Verify that the conditions for inference are satisfied.

- Compute and report the standard error, T-score, and p-value.

- Draw an appropriate conclusion regarding the hypotheses.

```
Hint: The `mature` variable takes on values of either `younger mom` or
`mature mom`. You can alter the code from Exercise 8 to perform the test!
```

#### State the hypotheses

- $H_0$:
- $H_A$:

#### Verify the conditions for inference

In [ ]:
# Fill in the code to make data sets for the younger moms and the mature moms.

younger_moms = nc[nc['mature'] == ???]
mature_moms = nc[nc['mature'] == ???]

In [ ]:
# Fill in the code to make histograms of `weeks` by `mature`.

(
    ggplot(nc) +
    aes(x = ???) +
    geom_histogram() +
    facet_wrap('mature')
)

In [ ]:
# Fill in the code to find the mean pregnancy length for each group.

mean_weeks_younger_moms = ???
mean_weeks_mature_moms = ???

print(f'Mean weeks for younger moms: {mean_weeks_younger_moms}')
print(f'Mean weeks for mature moms: {mean_weeks_mature_moms}')

In [ ]:
# Fill in the code to find the number of cases in each group.

n_younger_moms = ???
n_mature_moms = ???

print(f'Number of younger moms: {n_younger_moms}')
print(f'Number of mature moms: {n_mature_moms}')

In [ ]:
# Fill in the code to find the standard deviation of pregnancy length for each group.

sd_weeks_younger_moms = ???
sd_weeks_mature_moms = ???

print(f'Standard deviation of weeks for younger moms: {sd_weeks_younger_moms}')
print(f'Standard deviation of weeks for mature moms: {sd_weeks_mature_moms}')

In [ ]:
# After verifying the conditions for statistical inference, run the code below
# to run a T-test and interpret the results.

alpha = 0.05

degrees_of_freedom = min(n_younger_moms - 1, n_mature_moms - 1)

t_star = stats.t.ppf(1 - alpha / 2, degrees_of_freedom)

null_diff = 0
mean_diff = mean_weeks_mature_moms - mean_weeks_younger_moms

SE = np.sqrt(sd_weeks_younger_moms**2/n_younger_moms + sd_weeks_mature_moms**2/n_mature_moms)

T_score = (mean_diff - null_diff) / SE

p_value = 2 * (1 - stats.t.cdf(abs(T_score), degrees_of_freedom))

print(f'T-score = {T_score}')
print(f'p-value = {p_value}')

if p_value < alpha:
    print("Reject null hypothesis")
else:
    print("Fail to reject null hypothesis")

## ANOVA

We would expect that the length of pregnancy may be related to the birth weight. We can describe the `term` of a birth as follows.

- Extremely pre-term is a birth occurring in less than 28 weeks.

- Pre-term is a birth occurring between 28 and 37 weeks.

- Full-term is a birth occurring between 38 and 42 weeks.

- Post-term is a birth occuring in 43 or more weeks.

The code below will add a new categorical variable to the `nc` dataset for the `term` as described above.

In [ ]:
# Run this code to add a new variable `term` to the `nc` dataframe.

def determine_term(n):
  if n < 28:
    return 'x_pre'
  elif n < 38:
    return 'pre'
  elif n < 43:
    return 'full'
  elif n < 46:
    return 'post'

nc['term'] = [determine_term(nc['weeks'][i]) for i in range(len(nc['weeks']))]
nc.head()

Now, let's see if we can use ANOVA to determine a statistically significant difference in birth weights across the values of `term`. First, though, we should determine the number of cases in our sample for each `term`.

In [ ]:
# Run this code to determine the number of cases for each value of `term`.

nc['term'].value_counts()

There are only 7 cases in the `x_pre` (or extremely pre-term) category. With so few observations, this category may not reasonably meet the conditions for ANOVA, so we will proceed by comparing just the `pre`, `post`, and `full` groups, which all have at least 30 cases.

We'll now conduct ANOVA for the birth weights of the `pre`, `full`, and `post` term babies.

### Exercise 10

Identify the null and alternative hypotheses for ANOVA on the mean baby weights by `term` to deteremine if the mean baby weights is different across these groups.

- $H_0$:
- $H_A$:

### Exercise 11

Use visual diagnostics to try to verify that the conditions for inference are reasonably satisfied in this case. Recall, you need to verify

- independence,

- approximately normal, and

- constant variance.

Make note of any concerns you may have regarding these conditions being reasonably satisfied.

```
Hint: See Section 7.5 of the textbook for more detail on these conditions, if needed.
```

In [ ]:
# Run this code to examine histograms for the `pre`, `full`, and `post` groups to verify that there # are no particularly extreme outliers.

plot_data = nc[nc['term'].isin(['pre', 'full', 'post'])]

(
  ggplot(plot_data) +
  aes(x = plot_data['weight']) +
  geom_histogram(bins = 20) +
  facet_wrap('term')
)

In [ ]:
# Run this code to examine boxplots to investigate the constant variance condition.

(
  ggplot(plot_data) +
  aes(x = 'term', y = 'weight') +
  geom_boxplot()
)

A commonly used guideline for the constant variance condition is that, if the ratio of largest standard deviation to smallest deviation among the groups is between 0.5 and 2.0, it is reasonable to proceed with ANOVA. Run the code below to check this condition before proceeding.

In [ ]:
# Run this code to compare the largest and smallest standard deviations to see if ANOVA is
# reasonable.

pre_sd = nc[nc['term'] == 'pre']['weight'].std()
full_sd = nc[nc['term'] == 'full']['weight'].std()
post_sd = nc[nc['term'] == 'post']['weight'].std()

sds = [pre_sd, full_sd, post_sd]
sd_ratio = max(sds)/min(sds)

if (sd_ratio) < 0.5:
  print(f"Ratio of standard deviations: {sd_ratio}")
  print("Ratio is too small, do not proceed with ANOVA.")
elif (sd_ratio) > 2:
  print(f"Ratio of standard deviations: {sd_ratio}")
  print("Ratio is too large, do not proceed with ANOVA.")
else:
  print(f"Ratio of standard deviations: {sd_ratio}")
  print("Ratio is reasonable, proceed with ANOVA.")


### Exercise 12

- Compute and report the F-statistic.

- Draw an appropriate conclusion regarding the hypotheses.



In [ ]:
# Run this code to compute the F-statistic and its p-value, then interpret the result.

stats.f_oneway(
  nc[nc['term'] == 'pre']['weight'],
  nc[nc['term'] == 'full']['weight'],
  nc[nc['term'] == 'post']['weight']
)

---

This lab was adapted by Timothy L. Clark, derivative of [OpenIntro Statistics by Diez, Çetinkaya-Rundel, and Barr](https://www.openintro.org/book/os/), released under [Creative Commons BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/deed.en) license.